In [ ]:
import numpy as np
from scipy.optimize import minimize
import time

# --- CONSTANTES PHYSIQUES ---
n_qubits = 12       # Petit système -> simulation exacte possible
dim = 2**n_qubits   # 4096 états
p_layers = 1       # Profondeur du circuit (paramètres à optimiser)
k_sat = 8
ratio = 176.54      # Seuil de transition de phase
n_clauses = int(ratio * n_qubits)

print(f"--- PARTIE 1 : Optimisation des angles (Simulation n={n_qubits}) ---")

# --- 1. SOLVEUR PARFAIT & GÉNÉRATEUR D'INSTANCES ---

def generate_random_8sat(n, m, k=8):
    """Génère m clauses aléatoires (indices et signes)."""
    clauses = []
    for _ in range(m):
        vars_idx = np.random.choice(n, k, replace=False) # Indices
        signs = np.random.choice([-1, 1], k)             # Signes (-1 ou +1)
        clauses.append((vars_idx, signs))
    return clauses

def get_cost_diagonal(n, clauses):
    """
    Calcule le spectre complet de l'Hamiltonien classique (H_C).
    Retourne un vecteur de taille 2^n contenant le nombre de clauses violées pour chaque état.
    """
    cost_diag = np.zeros(2**n)
    
    # Brute force intelligent (vectorisé)
    # On génère la matrice de tous les bitstrings (taille 2^n x n)
    # indices de 0 à 2^n-1
    arange = np.arange(2**n)
    # Bitmasking pour extraire les bits
    # bits[i, j] est le j-ème bit de l'état i
    bits = ((arange[:, None] & (1 << np.arange(n)))) > 0
    spins = 1 - 2 * bits.astype(int) # Map 0->1, 1->-1
    
    for vars_idx, signs in clauses:
        # On extrait les spins des variables concernées par la clause
        # Shape: (2^n, k)
        relevant_spins = spins[:, vars_idx]
        
        # Une clause est satisfaite si au moins un spin match le signe
        # Elle est violée si TOUS les spins sont opposés au signe
        # signs shape: (k,) -> broadcasté
        violation_check = (relevant_spins != signs)
        
        # Si violation_check est True partout pour une ligne, la clause est violée
        clause_violee = np.all(violation_check, axis=1)
        
        cost_diag += clause_violee.astype(float)
        
    return cost_diag

print("Recherche d'une instance SATISFIABLE (Scan complet)...")
while True:
    # 1. Générer une instance aléatoire
    clauses = generate_random_8sat(n_qubits, n_clauses, k_sat)
    
    # 2. Scanner TOUTES les solutions (Solveur Parfait)
    H_C_diag = get_cost_diagonal(n_qubits, clauses)
    
    # 3. Vérifier l'énergie minimale
    min_energy = np.min(H_C_diag)
    
    # On compte le nombre de solutions exactes (E=0)
    n_solutions = np.sum(H_C_diag == 0)
    
    if min_energy == 0:
        print(f"-> Instance trouvée ! (Nombre de solutions exactes : {n_solutions})")
        break
    else:
        # Si min_energy > 0, c'est une instance UNSAT (impossible à résoudre)
        # On rejette et on recommence
        print(f"-> Instance UNSAT (Min Energy={min_energy}). Rejetée.")

# --- 2. SIMULATION QUANTIQUE EXACTE (QAOA) ---

# État initial |+> (superposition uniforme)
psi_0 = np.ones(dim, dtype=np.complex128) / np.sqrt(dim)

def qaoa_success_probability(angles):
    """
    Simule le circuit QAOA et retourne la probabilité de mesurer une solution parfaite.
    Angles: concaténation [beta_0...beta_p, gamma_0...gamma_p]
    """
    betas = angles[:p_layers]
    gammas = angles[p_layers:]
    
    state = psi_0.copy()
    
    for i in range(p_layers):
        beta = betas[i]
        gamma = gammas[i]
        
        # -- Phase Separator (H_C) --
        # Opérateur diagonal : exp(-i * gamma * E_x)
        state *= np.exp(-1j * gamma * H_C_diag)
        
        # -- Mixer (H_B) --
        # Rotation X sur tous les qubits : exp(-i * beta * Sum X)
        # Produit tensoriel de matrices 2x2.
        # Astuce : On utilise la propriété que RX agit indépendamment sur chaque qubit.
        # Code vectorisé rapide pour appliquer RX(2*beta) sur le tenseur d'état.
        
        c = np.cos(beta)
        s = -1j * np.sin(beta)
        
        # On reshape le vecteur d'état pour isoler chaque qubit j
        # C'est équivalent à appliquer la porte sur le j-ème fil
        for j in range(n_qubits):
            # Le vecteur est vu comme (2^(n-1-j), 2, 2^j)
            # L'axe 1 est le qubit j
            shape_before = (1 << (n_qubits - 1 - j), 2, 1 << j)
            state_reshaped = state.reshape(shape_before)
            
            # Application de la matrice [[c, s], [s, c]] sur l'axe 1
            # |0> -> c|0> + s|1>
            # |1> -> s|0> + c|1>
            psi_0_component = state_reshaped[:, 0, :]
            psi_1_component = state_reshaped[:, 1, :]
            
            new_0 = c * psi_0_component + s * psi_1_component
            new_1 = s * psi_0_component + c * psi_1_component
            
            state_reshaped[:, 0, :] = new_0
            state_reshaped[:, 1, :] = new_1
            
            # Pas besoin de reshape back explicite car numpy partage la mémoire
            # mais pour la clarté, le 'state' plat est modifié.
            
    # -- MESURE (Overlap avec le "Solveur Parfait") --
    probs = np.abs(state)**2
    
    # On somme les probabilités uniquement sur les indices identifiés comme solutions (E=0)
    # C'est ici qu'on utilise notre connaissance parfaite de la solution.
    p_success = np.sum(probs[H_C_diag == 0])
    
    return -p_success # Minimisation de l'opposé

# --- 3. OPTIMISATION CLASSIQUE ---

print("\nDémarrage de l'optimisation des angles (COBYLA)...")
t0 = time.time()

# Initialisation "Linear Ramp" (Adiabatique)
# Beta décroît (fort mixage au début -> faible à la fin)
# Gamma croît (faible coût au début -> fort à la fin)
dt = 0.7 
beta_init = [(p_layers - j) * dt * 0.05 for j in range(p_layers)]
gamma_init = [(j + 1) * dt * 0.05 for j in range(p_layers)]
x0 = np.array(beta_init + gamma_init)

# Optimisation
res = minimize(qaoa_success_probability, x0, method='COBYLA', options={'maxiter': 300, 'tol': 1e-4})

t_end = time.time()
print(f"Terminé en {t_end - t0:.2f}s")
print(f"Meilleure probabilité de succès (Overlap) : {-res.fun:.5f}")

# Extraction des résultats pour la partie 2
best_betas = res.x[:p_layers]
best_gammas = res.x[p_layers:]

print("\n--- RÉSULTATS À COPIER POUR PARTIE 2 ---")
print(f"best_betas = {list(best_betas)}")
print(f"best_gammas = {list(best_gammas)}")

--- PARTIE 1 : Optimisation des angles (Simulation n=12) ---
Recherche d'une instance SATISFIABLE (Scan complet)...
-> Instance UNSAT (Min Energy=1.0). Rejetée.
-> Instance UNSAT (Min Energy=1.0). Rejetée.
-> Instance trouvée ! (Nombre de solutions exactes : 2)

Démarrage de l'optimisation des angles (COBYLA)...
Terminé en 0.05s
Meilleure probabilité de succès (Overlap) : 0.00195

--- RÉSULTATS À COPIER POUR PARTIE 2 ---
best_betas = [np.float64(0.7123648700613924)]
best_gammas = [np.float64(1.9488733919317918)]


In [ ]:
import numpy as np

# Try importing tqdm for the progress bar, otherwise define a silent fallback
try:
    from tqdm import tqdm
except ImportError:
    def tqdm(iterable, desc=None):
        return iterable

def get_scaling_exponent_qaoa_ksat(r, gammas, betas, k, verbose=True):
    """
    Computes the scaling exponent of the QAOA success probability on random k-SAT.
    
    Implements Proposition 3 and Equation 85 (Source 1285) from Boulebnane & Montanaro (2022).
    By including all subsets (including singletons) in the index set A, the 
    fixed-point iteration captures the full scaling exponent (including the prefactor).
    
    Parameters:
    r (float): Clauses-to-variables ratio.
    gammas (list[float]): QAOA gamma angles.
    betas (list[float]): QAOA beta angles.
    k (int): k-SAT parameter (must be a power of 2, k=2^q).
    verbose (bool): If True, shows progress.
    
    Returns:
    float: The scaling exponent C (success prob ~ exp(C*n)).
    """
    
    # --- 1. Validation and Setup ---
    q_val = int(np.log2(k))
    if 2**q_val != k:
        raise ValueError("k must be a power of 2 (k=2^q).")
    
    p = len(gammas)
    dim = 2 * p + 1
    size = 1 << dim
    
    # --- 2. Compute Vectors b and c ---
    # We define b_s and c_alpha according to Example 16 / Equation 85.
    # Crucially, we do NOT mask singletons. The sum runs over ALL subsets alpha.
    
    s_indices = np.arange(size)
    s_bits = [(s_indices >> j) & 1 for j in range(dim)]
    
    # Vector b_s = B_{beta, s} / 2
    # The factor 1/2 absorbs the 1/2^n prefactor from the total probability definition.
    term_sign = (-1.0) ** (s_bits[0] ^ s_bits[p])
    prod_cos = np.ones(size, dtype=np.complex128)
    prod_sin = np.ones(size, dtype=np.complex128)
    
    for j in range(p):
        beta_val = betas[j]
        c_val = np.cos(beta_val / 2)
        s_val = 1j * np.sin(beta_val / 2)
        
        # Exponents for cos and sin based on bit comparisons (Eq 25)
        eq = (1 - (s_bits[j] ^ s_bits[j+1])) + (1 - (s_bits[2*p - j] ^ s_bits[2*p - j - 1]))
        neq = (s_bits[j] ^ s_bits[j+1]) + (s_bits[2*p - j] ^ s_bits[2*p - j - 1])
        
        prod_cos *= (c_val ** eq)
        prod_sin *= (s_val ** neq)
        
    b = (term_sign * prod_cos * prod_sin) / 2.0

    # Vector c_alpha (Eq 16 / 52)
    # We compute this for ALL alpha. The fixed point will naturally handle the linear 
    # terms (singletons) that correspond to the "prefactor".
    c = r * ((-1.0) ** s_bits[p]).astype(np.complex128)
    
    for j in range(dim):
        in_alpha = s_bits[j]
        term = 1.0
        if j < p:
            term = np.exp(-1j * gammas[j] / 2) - 1
        elif j > p:
            gamma_idx = 2 * p - j
            term = np.exp(1j * gammas[gamma_idx] / 2) - 1
        
        mask = (in_alpha == 1)
        c[mask] *= term

    # Note: We do NOT zero out singletons.
    # Precompute (-c)^(1/k)
    neg_c_pow = (-c) ** (1.0 / k)

    # --- 3. Efficient Summation Algorithms ---
    # These compute the matrix-vector products A*v and A^T*v.
    # Since A_{alpha, s} = 1/2 * 1[s compatible with alpha], we must divide by 2.
    
    def algorithm_3_sum_alpha(v):
        """ Computes X_s = sum_{alpha} A_{alpha, s} v_{alpha}. """
        n = dim
        z1 = v.copy()
        # Subset Sum
        for i in range(n):
            shape = (1 << (n - 1 - i), 2, 1 << i)
            z1r = z1.reshape(shape)
            z1r[:, 1, :] += z1r[:, 0, :]
            
        # Superset Sum (via reverse)
        indices = np.arange(size)
        rev_indices = indices ^ (size - 1)
        z0 = v[rev_indices].copy() 
        for i in range(n):
            shape = (1 << (n - 1 - i), 2, 1 << i)
            z0r = z0.reshape(shape)
            z0r[:, 0, :] += z0r[:, 1, :]
        z0 = z0[rev_indices]
        
        # Normalization by 1/2 for A matrix
        return (z0 + z1 - v[0]) / 2.0

    def algorithm_4_sum_s(v):
        """ Computes Y_alpha = sum_{s} A_{alpha, s} v_{s}. """
        # Symmetric to Algorithm 3
        n = dim
        indices = np.arange(size)
        rev_indices = indices ^ (size - 1)
        
        z0 = v[rev_indices].copy()
        z1 = v.copy()
        
        for i in range(n):
            shape = (1 << (n - 1 - i), 2, 1 << i)
            z0r = z0.reshape(shape)
            z0r[:, 0, :] += z0r[:, 1, :]
            z1r = z1.reshape(shape)
            z1r[:, 1, :] += z1r[:, 0, :]
            
        return (z0 + z1 - v[0]) / 2.0

    from scipy.optimize import root
    import warnings

    # --- 4. UNIVERSAL BROKEN-SYMMETRY ANSATZ (p >= 1) ---
    # The boolean state space dimension is 2^(2p+1).
    # At any depth p, the strictly violating history is either the all-0s or all-1s state.
    # We apply the massive O(10) polarization to the all-1s state (index -1) to anchor the physical branch.
    z_ansatz = np.full(size, 1.5 + 0.0j, dtype=np.complex128)
    z_ansatz[-1] = 15.0 * np.exp(1j * 0.8)  # The secondary critical point anchor

    # --- 5. THE PRX RESIDUAL MAPPING ---
    def residual(z_real_flat):
        z_curr = z_real_flat[:size] + 1j * z_real_flat[size:]

        # Forward pass through the PRX Cavity Equations
        V = neg_c_pow * z_curr
        X = algorithm_3_sum_alpha(V)
        X_max = np.max(X.real)

        # Stabilized exponentiation
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            E_stable = np.exp(X - X_max)

        weighted_E = b * E_stable
        D_stable = np.sum(weighted_E)

        # Guard against unphysical collapse
        if np.abs(D_stable) < 1e-15:
            return z_real_flat * 1e9

        Y_stable = algorithm_4_sum_s(weighted_E)
        grad_F = neg_c_pow * (Y_stable / D_stable)

        # The saddle map constraint
        z_next = -k * (grad_F ** (k - 1))

        diff = z_curr - z_next
        return np.concatenate([np.real(diff), np.imag(diff)])

    z_init_flat = np.concatenate([np.real(z_ansatz), np.imag(z_ansatz)])

    # --- 6. ADAPTIVE SOLVER ROUTING ---
    if verbose:
        print(f"\n[Depth p={p} | Variables: {size}]")
        print("Launching Newton solver targeting the physical broken-symmetry root...")

    # If the state space is small enough (e.g., p <= 3, size <= 128), Levenberg-Marquardt is mathematically superior.
    # If the state space is massive (p > 3), we MUST use Jacobian-free Krylov to prevent RAM exhaustion.
    solver_method = 'lm' if size <= 1024 else 'krylov'

    sol = root(
        residual,
        z_init_flat,
        method=solver_method,
        options={'fatol': 1e-8} if solver_method == 'krylov' else {'ftol': 1e-10}
    )

    if not sol.success:
        if verbose:
            print("Warning: Solver struggled to converge cleanly. Message:", sol.message)

    z_star = sol.x[:size] + 1j * sol.x[size:]

    # --- 7. FINAL SCALING EXPONENT & NORMALIZATION ---
    V = neg_c_pow * z_star
    X = algorithm_3_sum_alpha(V)
    X_max = np.max(X.real)
    E_stable = np.exp(X - X_max)

    weighted_E = b * E_stable
    D_stable = np.sum(weighted_E)
    Y_stable = algorithm_4_sum_s(weighted_E)

    # Apply the exact state-space normalization (1/2^q) required by the unconstrained partition function.
    # Because q = dim (which is 2p+1), we divide the sum by 2**dim.
    F_val = np.log(D_stable / (2**dim)) + X_max

    # Calculate edge penalties using the converged gradient
    grad_F = neg_c_pow * (Y_stable / D_stable)
    sum_grad_pow = np.sum(grad_F ** k)

    # Subtract the overlap penalty (with the r clause density if r is external, or (k-1) if r is absorbed into grad_F).
    # Note: If your grad_F absorbs r, use (k-1). If not, use r * (k-1).
    exponent = F_val - (k - 1) * sum_grad_pow

    return np.real(exponent)

In [ ]:
k = 8 
betas = best_betas
gammas = best_gammas
r = 176.54  
get_scaling_exponent_qaoa_ksat(r, gammas, betas, k)

Fixed-point iteration:   8%|▊         | 75/1000 [19:35<4:01:37, 15.67s/it]


5.030535795289101